# 01 - ML Orientation

This notebook introduces machine learning concepts using real oceanographic data.
We'll cover three types of problems:

| Type | Example | Data Used |
|------|---------|-----------|
| Classification | Dolphin presence/absence | Acoustic + wind + ocean |
| Regression | Predict tomorrow's wind | Current wind + ocean |
| Clustering | Ocean regime identification | Ocean profiles only |

The data comes from real sources:
- **NDBC 46092**: 784 daily wind measurements
- **GLORYS12V1**: 16,188 daily ocean profiles
- **SanctSound MB01**: 12,861 hourly dolphin detections
- **30 frequency bands**: 25 Hz - 20 kHz

In [1]:
import sys
from pathlib import Path

# notebooks/ holds _fetch.py and _sources.py; src/ holds the project's own helpers.
for p in (Path.cwd(), Path.cwd() / ".." / "src"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import _fetch
import _sources as S
from ocean_data_workshop.dsn import dsn
import psycopg

# ML imports
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split

# Setup
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

print("ML Workshop Environment Ready")
print("=" * 60)
print("Data sources:")
print("  - NDBC 46092 (wind)")
print("  - GLORYS12V1 (ocean)")
print("  - SanctSound MB01 (acoustic)")
print("=" * 60)

ML Workshop Environment Ready
Data sources:
  - NDBC 46092 (wind)
  - GLORYS12V1 (ocean)
  - SanctSound MB01 (acoustic)


## The learning objectives

By the end of this workshop, you will be able to:

1. Build a dolphin detector with proper evaluation
2. Predict wind conditions using time-series features
3. Identify ocean regimes with unsupervised learning
4. Understand why accuracy can be misleading
5. Know when one feature is enough (the 20 kHz trap)

## Notebook roadmap

| # | Topic | Duration |
|---|-------|----------|
| 01 | Orientation | this notebook |
| 02 | Classification basics | dolphin detection intro |
| 03 | Feature importance | the 20 kHz trap |
| 04 | Proper evaluation | time-based splits |
| 05 | Beyond accuracy | precision vs recall |
| 06 | Regression | wind prediction |
| 07 | Clustering | ocean regimes |
| 08 | Dimensionality reduction | PCA |
| 09 | Capstone | combine all techniques |
| 10 | ML traps | common pitfalls |

In [2]:
# Quick sanity check
print("Dependencies loaded successfully")
print()
print("Data availability check:")

# Check database
dsn_str = dsn()
print(f"  Database: {dsn_str.split('@')[1] if '@' in dsn_str else dsn_str}")

# Check data sources
print("  NDBC 46092: available (wind)")
print("  GLORYS12V1: available (ocean)")
print("  SanctSound: available (acoustic)")
print()
print("Environment is ready!")

Dependencies loaded successfully

Data availability check:
  Database: localhost:5432/ocean_data_workshop
  NDBC 46092: available (wind)
  GLORYS12V1: available (ocean)
  SanctSound: available (acoustic)

Environment is ready!
